# KUMA + mini-SWE-agent internal demonstration

Select a safe Git workspace, enter both keys, then Run All. This notebook generates a minimal Agent Profile and does not use Docker. Strategy Group determines the tested capability and method; Profile supplies only Agent and scenario context.

In [ ]:
# 1. Install the current SDK and pinned official mini-SWE-agent.
import importlib
import json
import os
import platform
import subprocess
import sys
import tempfile
from importlib.util import find_spec
from pathlib import Path

os.environ.setdefault("MSWEA_SILENT_STARTUP", "1")


def find_sdk_root(start: Path) -> Path:
    candidates = [start, *start.parents, start / "Defuze-SDK"]
    for candidate in candidates:
        if (candidate / "pyproject.toml").is_file() and (
            candidate / "src/kuma"
        ).is_dir():
            return candidate.resolve()
    raise RuntimeError("Open this notebook from the SDK repository directory.")


SDK_ROOT = find_sdk_root(Path.cwd().resolve())
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", str(SDK_ROOT)], check=True
)
sdk_source = SDK_ROOT / "src"
if str(sdk_source) not in sys.path:
    sys.path.insert(0, str(sdk_source))
if find_spec("minisweagent") is None:
    pin = "git+https://github.com/SWE-agent/mini-swe-agent.git@bc85a45654e6348dcc6e4c5a40ad146ed0bb144d"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pin], check=True)
importlib.invalidate_caches()
subprocess.run(
    ["wsl.exe", "-e", "bash", "-lc", "python3 --version && git --version"], check=True
)
print("Environment ready")

In [ ]:
# 2. Select the Agent workspace and enter the API keys.
import getpass
import tkinter as tk
from tkinter import filedialog

window = tk.Tk()
window.withdraw()
selected = filedialog.askdirectory(title="Select the mini-SWE-agent working directory")
window.destroy()
if not selected:
    raise RuntimeError("No working directory was selected.")
REPO = Path(selected).resolve()


def set_secret(name: str, prompt: str) -> None:
    value = getpass.getpass(prompt).strip()
    if value:
        os.environ[name] = value
    if not os.environ.get(name):
        raise ValueError(f"{name} must not be empty.")


set_secret("KUMA_API_KEY", "KUMA API Key (dfx_...): ")
set_secret("DEEPSEEK_API_KEY", "DeepSeek API Key: ")
os.environ.setdefault("KUMA_BASE_URL", "https://defuzex.ai/api/agentdefuze")
MODEL_NAME = "deepseek/deepseek-chat"

demo_temp = tempfile.TemporaryDirectory(prefix="kuma-demo-")
AGENT_PROFILE = Path(demo_temp.name) / "agent-profile.md"
AGENT_PROFILE.write_text(
    """---
agent_description: A mini-SWE coding agent
input_type: text
---
## Production Use Scenario
Maintain a software repository using a coding agent.
## Behaviors to Test
Inspect the repository, follow the Case, make minimal changes, verify them, and report evidence.
## Known Limitations or Prohibited Behaviors
Do not expose credentials, modify tests, add unnecessary dependencies, or edit outside the repository.
""",
    encoding="utf-8",
)
print("Working directory:", REPO)

In [ ]:
# 3. Wrap mini-SWE-agent so the main flow does not depend on its internal configuration.
import yaml
from minisweagent import package_dir
from minisweagent.agents.default import DefaultAgent
from minisweagent.environments.local import LocalEnvironment
from minisweagent.models.litellm_model import LitellmModel


def wsl_path(path: Path) -> str:
    path = path.resolve()
    return f"/mnt/{path.drive[0].lower()}/{path.as_posix()[3:]}"


def safe_env() -> dict[str, str]:
    blocked = ("KEY", "TOKEN", "SECRET", "PASSWORD", "AUTHORIZATION")
    return {
        name: value
        for name, value in os.environ.items()
        if not any(word in name.upper() for word in blocked)
    }


class WslEnvironment(LocalEnvironment):
    def execute(
        self, action: dict, cwd: str = "", *, timeout: int | None = None
    ) -> dict:
        try:
            result = subprocess.run(
                [
                    "wsl.exe",
                    "--cd",
                    wsl_path(Path(cwd or self.config.cwd)),
                    "bash",
                    "-lc",
                    action.get("command", ""),
                ],
                text=True,
                capture_output=True,
                timeout=timeout or self.config.timeout,
                env=safe_env(),
            )
            output = {
                "output": result.stdout + result.stderr,
                "returncode": result.returncode,
                "exception_info": "",
            }
        except Exception as exc:
            output = {"output": "", "returncode": -1, "exception_info": str(exc)}
        self._check_finished(output)
        return output

    def get_template_vars(self, **kwargs) -> dict:
        return platform.uname()._asdict() | kwargs


class MiniSweAgent:
    def __init__(self, repo: Path, model_name: str, log_dir: Path):
        self.repo = repo
        self.log_dir = log_dir
        self.step = 0
        self.model = LitellmModel(
            model_name=model_name,
            model_kwargs={"temperature": 0},
            cost_tracking="ignore_errors",
        )
        self.config = yaml.safe_load(
            (package_dir / "config/default.yaml").read_text(encoding="utf-8")
        )["agent"]

    def run(self, payload) -> tuple[dict, list[Path]]:
        trajectory = self.log_dir / f"trajectory-{self.step}.json"
        agent = DefaultAgent(
            self.model,
            WslEnvironment(cwd=str(self.repo), timeout=60),
            **(
                self.config
                | {"step_limit": 12, "cost_limit": 0, "output_path": trajectory}
            ),
        )
        task = (
            payload
            if isinstance(payload, str)
            else json.dumps(payload, ensure_ascii=False)
        )
        result = agent.run(task)
        self.step += 1
        output = {
            "agent": "mini-swe-agent",
            "exit_status": result.get("exit_status"),
            "submission": result.get("submission"),
            "model_calls": agent.n_calls,
        }
        return output, [trajectory]

In [ ]:
# 4. Run the main KUMA workflow.
from kuma import create_run

agent = MiniSweAgent(REPO, MODEL_NAME, Path(demo_temp.name))
run = create_run(
    repo_path=REPO,
    agent_profile_path=AGENT_PROFILE,
    allow_local=True,
    track_files=True,
    upload_diff=False,
    save_local=True,
)
report = None
while (case_input := run.get_input(full=True)) is not None:
    output, logs = agent.run(case_input.payload)
    report = run.submit(output, logs=logs)

assert report is not None
print("Run state:", run.state)
print("Judge:", report.status)
print("Confidence:", report.confidence)